## 函式庫

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import clip
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset, random_split
from PIL import Image
import pandas as pd
import numpy as np
import os

## 下載資料集

In [16]:
#!/bin/bash
! kaggle datasets download darkfanxing/ntutemnist

Dataset URL: https://www.kaggle.com/datasets/darkfanxing/ntutemnist
License(s): unknown
100%|███████████████████████████████████████▉| 305M/306M [00:12<00:00, 30.0MB/s]
100%|████████████████████████████████████████| 306M/306M [00:12<00:00, 26.3MB/s]


In [ ]:
! unzip ntutemnist.zip -d ntutemnist_data

Archive:  ntutemnist.zip
replace ntutemnist_data/emnist-byclass-test.npz? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

## 檢查資料

In [ ]:
DATA_DIR = 'ntutemnist_data/'
TRAIN_DATA_FILE = DATA_DIR + 'emnist-byclass-train.npz'
TEST_DATA_FILE = DATA_DIR + 'emnist-byclass-test.npz'
train_data = np.load(TRAIN_DATA_FILE)
test_data = np.load(TEST_DATA_FILE)
train_images = train_data['training_images']
train_labels = train_data['training_labels']
test_images = test_data['testing_images']

## Model引入

In [ ]:
# 設置設備
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device}")

# 載入 CLIP 模型，這裡我們使用 'ViT-B/32'
clip_model, preprocess = clip.load("ViT-B/32", device=device)

# 凍結 CLIP 權重
for param in clip_model.parameters():
    param.requires_grad = False

Using cuda


In [ ]:
print("Train Images Shape:", train_images.shape)  # (N, 28, 28) 或 (N, 1, 28, 28)
print("Train Labels Shape:", train_labels.shape)  # (N,)
print("Test Images Shape:", test_images.shape)    # (M, 28, 28) 或 (M, 1, 28, 28)

import matplotlib.pyplot as plt

def show_images(images, labels, num=5):
    plt.figure(figsize=(10, 5))
    for i in range(num):
        plt.subplot(1, num, i+1)
        plt.imshow(images[i], cmap='gray')  # 確保影像是灰階
        plt.title(f"Label: {labels[i]}")
        plt.axis("off")
    plt.show()

show_images(train_images, train_labels, num=5)



## 數據預處理

In [14]:
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # CLIP 需要 RGB
    transforms.Resize((224, 224)),  
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.481, 0.457, 0.408], std=[0.268, 0.261, 0.275])  
])

test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.481, 0.457, 0.408], std=[0.268, 0.261, 0.275])
])


## 自定義 Dataset

In [ ]:
class EMNISTDataset(Dataset):
    def __init__(self, npz_path, transform=None, has_labels=True):
        data = np.load(npz_path)
        self.images = data["training_images"] if has_labels else data["testing_images"]
        self.labels = data["training_labels"] if has_labels else None
        self.transform = transform
        self.has_labels = has_labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = Image.fromarray(self.images[idx].reshape(28, 28).astype(np.uint8))
        if self.transform:
            image = self.transform(image)

        if self.has_labels:
            label = int(self.labels[idx])
            return image, label
        return image


## 定義 CLIP + MLP

In [ ]:
class CLIP_EMNIST_Model(nn.Module):
    def __init__(self, clip_model, num_classes=62):
        super(CLIP_EMNIST_Model, self).__init__()
        self.clip_visual = clip_model.visual  # CLIP 的影像 encoder
        self.fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)  # 分類 62 個類別
        )

    def forward(self, x):
        with torch.no_grad():  # 凍結 CLIP
            features = self.clip_visual(x)
        return self.fc(features)


## 訓練 & 驗證

In [ ]:
# 讀取數據
train_dataset = EMNISTDataset("/kaggle/input/ntutemnist/emnist-byclass-train.npz", transform=train_transform)
test_dataset = EMNISTDataset("/kaggle/input/ntutemnist/emnist-byclass-test.npz", transform=test_transform, has_labels=False)

# 切割 10% 訓練集作為驗證集
val_size = int(0.1 * len(train_dataset))
train_size = len(train_dataset) - val_size
train_data, val_data = random_split(train_dataset, [train_size, val_size])

# DataLoader
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 初始化模型
model = CLIP_EMNIST_Model(clip_model).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.0001)

# 訓練
def train(model, train_loader, val_loader, epochs=10):
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        val_acc = evaluate(model, val_loader)
        print(f"Epoch {epoch+1}/{epochs}: Loss={total_loss/len(train_loader):.4f}, Train Acc={correct/total:.4f}, Val Acc={val_acc:.4f}")

def evaluate(model, val_loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return correct / total

# 開始訓練
train(model, train_loader, val_loader, epochs=10)


## 測試 & 預測

In [ ]:
def predict(model, test_loader):
    model.eval()
    predictions = []
    with torch.no_grad():
        for images in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            predictions.extend(predicted.cpu().numpy())

    return predictions

# 預測
test_predictions = predict(model, test_loader)

# 儲存結果
pd.DataFrame({"Predicted": test_predictions}).to_csv("/kaggle/working/emnist_predictions.csv", index=False)
print("預測結果已儲存至 emnist_predictions.csv")
